# Data Science project 2026-27 
**MSc Supply Chain & Operations Management -- EMLyon Business School**

You work in pairs on **two of the three situations** below (your choice). Each situation reproduces a problem you are
likely to meet in your first job: the data are generated by the cell *Data generators* and are **the same for every
student** (fixed random seed). Do not modify that cell.

| Situation | Company (fictitious) | Problem | Type |
|---|---|---|---|
| 1 | Modalia, online fashion & home retailer | Which orders will be returned, and which ones deserve a "fit check" before shipping? | classification + cost |
| 2 | Logivia, third-party logistics warehouse | How long will each picking task take, and how many pickers should be planned per shift? | regression + staffing |
| 3 | Rhodanis Parcel Hub | Which conveyor motors will fail within 14 days, and which 4 should be inspected each week? | rare events + capacity |

**Method**: the 8 steps of the final project of the course notebook (Part 14) -- explore, prepare, benchmark, improve the
variables, tune, decide and test once, explain, conclude. Reuse the code of the course notebook: everything you need is there.

**How to read this notebook.** Cells marked **(given)** load the data and provide small helpers. Cells marked **EXAMPLE**
start the work on a minimal case: extend or replace them. Headings **Step 1 ... Step 8** follow the brief; the questions in
italics are there to guide you, not to limit you. Delete the situation you do not choose.

Before handing in: *Kernel > Restart & Run All* must run without error.

In [ ]:
# ---- Libraries (run this cell) -- same conventions as the course notebook ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import time, warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import (train_test_split, cross_validate, cross_val_predict, KFold, StratifiedKFold,
                                     GroupKFold, TimeSeriesSplit, GridSearchCV)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor, AdaBoostClassifier,
                              GradientBoostingClassifier, GradientBoostingRegressor)
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix, mean_absolute_error,
                             root_mean_squared_error, r2_score)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

# Colours used everywhere: BLUE = "normal", ORANGE = "problem" (return, failure, ...)
BLUE, ORANGE, AQUA, GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8985"
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 11})
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 3)
RANDOM_STATE = 42

## Data generators (given)
As in the course notebook, the data are **synthetic** but built to mimic real mechanisms (customer habits, learning curves
of new employees, wear of the equipment...). The teaching team knows the rules that generated them.
**There is no need to spend time on how the data are generated, since most of the work consist in deploying and comparing the models studied in class**

In [ ]:
# ---- Data generators of the project (run this cell; do NOT read it before the end: it contains the hidden rules) ----
import numpy as np
import pandas as pd


def _sigmoid(z):
    return 1 / (1 + np.exp(-z))


def make_returns(n=24000, seed=2027):
    """Situation 1 -- online orders of a fashion & home retailer; target: was the order returned?"""
    rng = np.random.default_rng(seed)
    days = pd.date_range("2025-01-01", "2026-10-31", freq="D")
    season = 1 + 0.35 * np.isin(days.month, [11, 12]) + 0.15 * np.isin(days.month, [1, 7])      # sales peaks
    order_date = rng.choice(days, n, p=season / season.sum())
    order_date = np.sort(order_date)
    month = pd.DatetimeIndex(order_date).month
    category = rng.choice(["apparel", "shoes", "home", "beauty", "electronics"], n, p=[.42, .16, .20, .12, .10])
    is_fashion = np.isin(category, ["apparel", "shoes"])
    n_items = np.clip(rng.poisson(np.where(is_fashion, 2.2, 1.4)), 1, 12)
    unit_price = rng.lognormal(np.select([category == "apparel", category == "shoes", category == "home",
                                          category == "beauty"], [3.4, 4.1, 3.6, 2.9], 4.7), 0.45)
    order_value = (n_items * unit_price).round(2)
    sale_period = np.isin(month, [1, 7]) | ((month == 11) & (pd.DatetimeIndex(order_date).day >= 24))
    discount = np.where(sale_period, rng.choice([20, 30, 40, 50], n, p=[.3, .35, .25, .1]),
                        rng.choice([0, 0, 0, 10, 15, 20], n)).astype(int)
    size_dup = np.where(is_fashion & (n_items >= 2), rng.binomial(np.maximum(n_items - 1, 0), 0.22), 0)
    tenure = np.where(rng.random(n) < 0.14, 0, np.clip(rng.gamma(1.4, 14, n), 1, 120)).round(0)
    new_customer = tenure < 1
    past_orders = np.where(new_customer, 0, np.clip(rng.poisson(1 + tenure / 6), 1, 80))
    habit = np.clip(rng.beta(2, 5, n) + 0.12 * is_fashion, 0, 1)                         # hidden return habit
    past_rr = np.where(new_customer, np.nan, np.clip(rng.binomial(np.maximum(past_orders, 1), habit) /
                                                     np.maximum(past_orders, 1), 0, 1)).round(2)
    device = rng.choice(["mobile_app", "mobile_web", "desktop"], n, p=[.38, .34, .28])
    payment = rng.choice(["card", "paypal", "buy_now_pay_later", "gift_card"], n, p=[.55, .25, .15, .05])
    loyalty = (rng.random(n) < np.clip(0.15 + tenure / 120, 0, 0.75)).astype(int)
    policy_change = pd.Timestamp("2026-03-01")
    free_returns = np.where(order_date < np.datetime64(policy_change), 1, loyalty)          # policy change in March 2026
    size_guide = (rng.random(n) < np.where(is_fashion, 0.35, 0.05)).astype(int)
    region = rng.choice(["Ile-de-France", "Auvergne-Rhone-Alpes", "Occitanie", "Hauts-de-France", "Other"], n,
                        p=[.27, .14, .10, .09, .40])
    promised = rng.choice([2, 3, 5], n, p=[.25, .55, .20])
    # ---------- hidden rule ----------
    z = (-2.75 + pd.Series(category).map({"apparel": 1.05, "shoes": 0.85, "home": -0.55, "beauty": -1.6,
                                           "electronics": -0.3}).values
         + 1.7 * (size_dup > 0)                                        # "bracketing": several sizes of the same item
         + 0.30 * np.log(n_items)
         + 0.025 * np.maximum(discount - 20, 0)                        # deep discounts = impulse purchases
         + np.where(new_customer, 0.35, 5.0 * (habit - 0.3))           # customer habit (seen through past_return_rate)
         + 0.45 * (payment == "buy_now_pay_later") - 0.6 * (payment == "gift_card")
         + 0.30 * (device == "mobile_web")
         - 0.65 * size_guide * is_fashion
         + 0.55 * free_returns
         + 0.002 * np.minimum(order_value, 400))
    delivery_delay = np.clip(rng.poisson(0.6, n) + (rng.random(n) < 0.06) * rng.integers(2, 7, n), 0, 10)
    z = z + 0.18 * delivery_delay                                     # late delivery -> more returns (known AFTER delivery)
    returned = (rng.random(n) < _sigmoid(z)).astype(int)
    contacts = rng.poisson(0.15 + 0.9 * returned + 0.1 * delivery_delay)                   # after delivery (leakage)
    df = pd.DataFrame(dict(order_id=np.arange(100001, 100001 + n), order_date=pd.to_datetime(order_date),
                           category=category, n_items=n_items, order_value_eur=order_value, discount_pct=discount,
                           n_size_duplicates=size_dup, customer_tenure_months=tenure, customer_past_orders=past_orders,
                           customer_past_return_rate=past_rr, device=device, payment_method=payment,
                           loyalty_member=loyalty, free_returns=free_returns, viewed_size_guide=size_guide,
                           region=region, promised_delivery_days=promised, delivery_delay_days=delivery_delay,
                           customer_service_contacts=contacts, returned=returned))
    return df


def make_picking(n=45000, seed=2028):
    """Situation 2 -- picking tasks in a 3PL warehouse; target: picking time (minutes)."""
    rng = np.random.default_rng(seed)
    n_pickers = 110
    hired = rng.gamma(1.6, 14, n_pickers)                                   # experience at the start (months)
    hired[rng.choice(n_pickers, 25, replace=False)] = rng.uniform(0, 1.5, 25)  # seasonal newcomers
    days = pd.bdate_range("2026-03-02", periods=130)                         # 26 weeks, Monday-Friday
    day = rng.choice(len(days), n)
    day.sort()
    date = days[day]
    shift = rng.choice(["morning", "afternoon", "night"], n, p=[.42, .38, .20])
    hour = np.select([shift == "morning", shift == "afternoon"], [rng.integers(6, 14, n), rng.integers(14, 22, n)],
                     (22 + rng.integers(0, 8, n)) % 24)
    picker = rng.integers(0, n_pickers, n)
    experience = (hired[picker] + day / 21.7).round(1)
    order_type = rng.choice(["e-commerce", "store", "b2b"], n, p=[.55, .30, .15])
    n_lines = np.clip(np.select([order_type == "e-commerce", order_type == "store"],
                                [rng.poisson(2.5, n) + 1, rng.poisson(14, n) + 3], rng.poisson(30, n) + 5), 1, 120)
    n_units = n_lines * np.clip(np.select([order_type == "e-commerce", order_type == "store"],
                                          [rng.poisson(0.4, n) + 1, rng.poisson(3, n) + 1], rng.poisson(9, n) + 1), 1, None)
    n_zones = np.clip(np.round(1 + 6 * (1 - np.exp(-n_lines / 9)) + rng.normal(0, 0.6, n)), 1, 8).astype(int)
    weight = np.clip(n_units * rng.lognormal(-0.3, 0.8, n), 0.1, 900).round(1)
    cold = (rng.random(n) < np.where(order_type == "store", 0.35, 0.08)).astype(int)
    high_lines = rng.binomial(n_lines, 0.12)
    equipment = np.where(weight > 250, "pallet_jack", np.where(high_lines > 3, "reach_truck", "cart"))
    occupancy = np.clip(72 + 10 * np.sin(2 * np.pi * day / 65) + rng.normal(0, 4, n), 55, 98).round(1)
    new_wms = (date >= pd.Timestamp("2026-06-08")).astype(int)               # new routing software from week 15
    # ---------- hidden rule ----------
    travel = 1.1 * n_zones * (1 - 0.18 * new_wms) * (1 + 0.12 * np.maximum(occupancy - 85, 0))   # congestion > 85 %
    handling = 0.30 * n_lines + 0.022 * n_units + (0.012 * weight + 0.05 * np.maximum(weight - 120, 0)) * (equipment == "cart")
    lifts = 0.85 * high_lines * np.where(equipment == "reach_truck", 0.5, 1.0)
    base = 1.5 + travel + handling + lifts + 2.2 * cold
    learn = 1 + 0.9 * np.exp(-experience / 2.5)                               # learning curve of new pickers
    night = np.where(shift == "night", 1.08, 1.0)
    minutes = base * learn * night * rng.lognormal(0, 0.16, n)
    interrupt = rng.random(n) < 0.025
    minutes = minutes + interrupt * rng.uniform(8, 35, n)                    # interruptions (outliers)
    minutes = minutes.round(2)
    df = pd.DataFrame(dict(task_id=np.arange(1, n + 1), date=date, shift=shift, hour=hour,
                           picker_id=[f"P{p:03d}" for p in picker], picker_experience_months=experience,
                           order_type=order_type, n_lines=n_lines, n_units=n_units, n_zones=n_zones,
                           total_weight_kg=weight, cold_items=cold, high_shelf_lines=high_lines, equipment=equipment,
                           warehouse_occupancy_pct=occupancy, new_wms=new_wms,
                           lines_per_minute=(n_lines / minutes).round(3),                  # computed AFTER picking
                           pick_minutes=minutes))
    return df


def make_maintenance(n_machines=90, n_weeks=78, seed=2029):
    """Situation 3 -- weekly sensor summaries of the conveyor motors of a parcel hub; target: failure within 14 days."""
    rng = np.random.default_rng(seed)
    weeks = pd.date_range("2025-06-02", periods=n_weeks, freq="7D")
    woy = weeks.isocalendar().week.to_numpy()
    peak = np.isin(woy, list(range(46, 53)) + [1])                           # Black Friday & Christmas
    ambient = 14 + 9 * np.sin(2 * np.pi * (woy - 16) / 52)
    rows = []
    for m in range(n_machines):
        model = rng.choice(["M-200", "M-300", "M-450"], p=[.4, .35, .25])
        zone = rng.choice(["inbound", "sorter", "outbound"], p=[.3, .45, .25])
        age = rng.uniform(6, 110)
        wear = rng.uniform(0, 0.5)
        since_maint = rng.uniform(0, 1500)
        rate = {"M-200": 1.0, "M-300": 0.8, "M-450": 1.35}[model] * {"inbound": 0.9, "sorter": 1.2, "outbound": 1.0}[zone]
        down = 0
        for t in range(n_weeks):
            if down > 0:
                down -= 1
                continue
            load = np.clip(rng.normal(0.72 + 0.18 * peak[t], 0.07), 0.3, 1.0)
            hours = np.clip(rng.normal(118 + 30 * peak[t], 8), 60, 168)
            wear += rate * hours / 1000 * 0.17 * (1 + 1.2 * max(load - 0.8, 0) / 0.2) * (1 + age / 200)
            since_maint += hours
            vib = {"M-200": 1.8, "M-300": 2.9, "M-450": 1.3}[model] + 3.2 * wear ** 1.6 + 1.0 * load + rng.normal(0, 0.25)
            temp = 48 + 0.55 * ambient[t] + 9 * wear + 6 * load + rng.normal(0, 2.0)
            current = 11 + 4 * load + 1.6 * wear + rng.normal(0, 0.5)
            stops = rng.poisson(0.3 + 2.0 * max(wear - 0.6, 0))
            p_fail = _sigmoid(-8.4 + 6.0 * wear + 1.2 * (temp > 72) + 1.0 * (model == "M-450") * (load > 0.85))
            fail = rng.random() < p_fail
            rows.append(dict(machine_id=f"CV-{m + 1:02d}", week=weeks[t], model=model, zone=zone,
                             age_months=round(age + t * 7 / 30.4, 1), running_hours=round(hours, 1),
                             avg_load_pct=round(100 * load, 1), vibration_rms_mm_s=round(vib, 2),
                             motor_temp_max_c=round(temp, 1), current_draw_a=round(current, 2),
                             micro_stops=stops, hours_since_maintenance=round(since_maint),
                             ambient_temp_c=round(ambient[t] + rng.normal(0, 1.5), 1), _fail=int(fail)))
            if fail:                                                          # repair: reset of the wear
                wear, since_maint, down = rng.uniform(0, 0.15), 0, 1
            elif since_maint > 2000 and rng.random() < 0.5:                   # preventive maintenance
                wear, since_maint = wear * 0.5, 0
    df = pd.DataFrame(rows).sort_values(["machine_id", "week"]).reset_index(drop=True)
    g = df.groupby("machine_id")["_fail"]
    df["failure_next_14d"] = ((g.shift(-1) == 1) | (g.shift(-2) == 1)).astype(int)
    last = df.groupby("machine_id").cumcount(ascending=False) < 2
    df = df[~last].copy()                                                       # future not observed
    df["maintenance_ticket_next_week"] = ((g.shift(-1) == 1).loc[df.index] & (rng.random(len(df)) < 0.9)).astype(int)  # leakage
    df = df.drop(columns="_fail").reset_index(drop=True)
    return df

In [ ]:
returns = make_returns()          # Situation 1
picking = make_picking()          # Situation 2
motors = make_maintenance()       # Situation 3
for name, df in [("returns", returns), ("picking", picking), ("motors", motors)]:
    print(f"{name:8s} {df.shape[0]:>6,} rows x {df.shape[1]} columns")

## Helpers
`preprocess` builds the encoding + scaling step of a pipeline; `benchmark` cross-validates several models with the same folds (as in Part 10.6 of the course).

In [ ]:
# ---- Two helpers (run this cell) ----
def preprocess(cat_cols, num_cols):
    """One-hot encoding of the categorical columns + standardisation of the numeric ones (fitted on the training data only)."""
    return ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), list(cat_cols)),
                              ("num", StandardScaler(), list(num_cols))])


def benchmark(models, X, y, cv, scoring, groups=None):
    """Cross-validates every model with the SAME folds and returns mean +- std of each metric (cf. Part 10.6 of the course).
    models: dict name -> estimator or pipeline; scoring: list of scikit-learn metric names."""
    rows = []
    for name, model in models.items():
        t0 = time.time()
        res = cross_validate(model, X, y, cv=cv, scoring=scoring, groups=groups, n_jobs=-1)
        row = {"model": name}
        for s in scoring:
            vals = res[f"test_{s}"]
            sign = -1 if s.startswith("neg_") else 1
            row[s.replace("neg_", "")] = f"{sign * vals.mean():.3f} \u00b1 {vals.std():.3f}"
        row["time (s)"] = round(time.time() - t0, 1)
        rows.append(row)
    return pd.DataFrame(rows).set_index("model")

---
# Situation 1 -- Modalia: returned orders
Modalia sells clothes, shoes, home items, beauty products and small electronics online. One order in three is returned,
which costs about **16 EUR** per return (return shipping, handling, depreciation of the item). The customer-care team can
trigger a **fit check** before shipping (size advice by chat, exchange offer): it costs **1.20 EUR** per order and reduces the
probability of a return by **30 %**. Which orders should receive it? (details and column descriptions: see the brief)

#### Step 1 -- Explore (EXAMPLE to extend)

In [ ]:
display(returns.head())
print(f"Share of returned orders: {returns['returned'].mean():.1%}")
display(returns.describe().T[["mean", "std", "min", "max"]].round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
(returns.groupby("category")["returned"].mean() * 100).sort_values().plot.bar(ax=axes[0], color=ORANGE, rot=0)
axes[0].set_title("% of returned orders by category"); axes[0].set_ylabel("% returned"); axes[0].set_xlabel("")
monthly = returns.set_index("order_date")["returned"].resample("MS").mean() * 100
axes[1].plot(monthly.index, monthly.values, marker="o", color=ORANGE)
axes[1].set_title("% of returned orders by month"); axes[1].set_ylabel("% returned")
plt.tight_layout(); plt.show()

*What is the return rate by category, by customer profile, over time? Which variables seem related to returns?
Something changed in March 2026: what, and what are the consequences for your model?*

In [ ]:
# your code

#### Step 2 -- Prepare (EXAMPLE to extend)
*For each column: is it known **when the order is placed**? (two columns are only known after the delivery -- find them and
explain why they must not be used). How do you handle the missing values of `customer_past_return_rate`, and what do they mean?*

In [ ]:
# Chronological split: the model will be used on FUTURE orders -> train on the past, test on the last 4 months
train_r = returns[returns["order_date"] < "2026-07-01"].copy()
test_r = returns[returns["order_date"] >= "2026-07-01"].copy()      # locked away until Step 6
print(f"train: {len(train_r):,} orders  |  test: {len(test_r):,} orders")

# EXAMPLE of a first set of features -- Step 2: which columns are really known when the order is placed?
CAT1 = ["category", "device", "payment_method"]
NUM1 = ["n_items", "order_value_eur", "discount_pct", "n_size_duplicates"]

In [ ]:
# your code

#### Step 3 -- Benchmark (EXAMPLE to extend)
*Compare a naive baseline and **all** the models of the course with the same cross-validation. Which metric matters here?*

In [ ]:
# EXAMPLE of a benchmark with two models -- add the other models of the course (Step 3)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
models_1 = {
    "Logistic regression": make_pipeline(preprocess(CAT1, NUM1), LogisticRegression(max_iter=1000)),
    "Decision tree (depth 4)": make_pipeline(preprocess(CAT1, NUM1), DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)),
}
benchmark(models_1, train_r[CAT1 + NUM1], train_r["returned"], cv, scoring=["roc_auc", "recall", "precision"])

In [ ]:
# your code

#### Step 4 -- Improve the variables
*Can you create a business variable that helps the models (e.g. the share of duplicated sizes in the order)?*

In [ ]:
# your code

#### Step 5 -- Tune
*Tune the 1-2 most promising models with `GridSearchCV` on the training set only.*

In [ ]:
# your code

#### Step 6 -- Decide and test (once)
*Choose the decision rule from the business costs (helper below), not with a 0.5 threshold. Compare it with "never act" and "always act". Then evaluate **once** on the test set: cost per 1,000 orders, number of fit checks.*

In [ ]:
# your code

In [ ]:
# Business cost of a decision rule (Step 6): act = 1 if the fit-check is triggered for the order
COST_RETURN, COST_ACTION, ACTION_EFFECT = 16.0, 1.2, 0.30

def returns_cost(y_true, act):
    """Total cost (EUR) = cost of the actions + cost of the returns that still happen."""
    y_true, act = np.asarray(y_true), np.asarray(act).astype(bool)
    return float(np.where(act, COST_ACTION + (1 - ACTION_EFFECT) * y_true * COST_RETURN, y_true * COST_RETURN).sum())

y = train_r["returned"]
print(f"cost per 1,000 orders if we never act : {1000 * returns_cost(y, np.zeros(len(y))) / len(y):,.0f} EUR")
print(f"cost per 1,000 orders if we always act: {1000 * returns_cost(y, np.ones(len(y))) / len(y):,.0f} EUR")

In [ ]:
# your code

#### Step 7 -- Explain
*Which variables drive the returns (permutation importance, partial dependence)? What would you tell the customer-care manager?*

In [ ]:
# your code

#### Step 8 -- Conclude
*Read the generator `make_returns`: did your models find the hidden rules? What are the limits of your study?*

In [ ]:
# your code

---
# Situation 2 -- Logivia: picking time and staffing
In Logivia's warehouse, each order is prepared ("picked") by one operator. The site manager plans every evening the number
of pickers of each shift for the next day, from the list of orders to prepare. A picker provides **420 productive minutes**
per shift; a missing hour is paid **38 EUR** (overtime, temporary staff), an idle hour **24 EUR**. Today the plan uses the
average workload of each shift. Can a model of the picking time do better?

#### Step 1 -- Explore (EXAMPLE to extend)

In [ ]:
display(picking.head())
display(picking.describe().T[["mean", "std", "min", "max"]].round(2))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(picking["pick_minutes"], bins=60, color=BLUE, edgecolor="white")
axes[0].set_title("Distribution of the picking time"); axes[0].set_xlabel("minutes")
sns.boxplot(data=picking, x="order_type", y="pick_minutes", color=BLUE, fliersize=1, ax=axes[1])
axes[1].set_title("Picking time by order type"); axes[1].set_xlabel("")
axes[2].scatter(picking["n_lines"], picking["pick_minutes"], s=3, alpha=.3, color=BLUE)
axes[2].set_title("Picking time vs number of lines"); axes[2].set_xlabel("n_lines"); axes[2].set_ylabel("minutes")
plt.tight_layout(); plt.show()

*How is the picking time distributed (outliers?)? How does it depend on the size of the order, the zones visited, the
equipment, the shift, the experience of the picker? Something changed in June 2026: what?*

In [ ]:
# your code

#### Step 2 -- Prepare (EXAMPLE to extend)
*One column is computed **after** the picking from the target itself: which one? What do you do with the very long tasks?*

In [ ]:
# Chronological split: the last 6 weeks (30 working days) are the test period, locked away until Step 6
last_days = np.sort(picking["date"].unique())[-30:]
train_p = picking[~picking["date"].isin(last_days)].copy()
test_p = picking[picking["date"].isin(last_days)].copy()
print(f"train: {len(train_p):,} tasks ({train_p['date'].min():%d %b} - {train_p['date'].max():%d %b %Y})  |  "
      f"test: {len(test_p):,} tasks")

# EXAMPLE of a first set of features
CAT2 = ["shift", "order_type", "equipment"]
NUM2 = ["n_lines", "n_units", "n_zones", "total_weight_kg"]

In [ ]:
# your code

#### Step 3 -- Benchmark (EXAMPLE to extend)
*Compare the baseline and **all** the regression models of the course (MAE, RMSE, R²) with the same cross-validation.*

In [ ]:
# EXAMPLE: a naive baseline and a linear regression -- add the other models of the course (Step 3)
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
models_2 = {
    "Baseline (mean)": DummyRegressor(strategy="mean"),
    "Linear regression": make_pipeline(preprocess(CAT2, NUM2), LinearRegression()),
}
benchmark(models_2, train_p[CAT2 + NUM2], train_p["pick_minutes"], cv,
          scoring=["neg_mean_absolute_error", "neg_root_mean_squared_error", "r2"])

In [ ]:
# your code

#### Step 4 -- Improve the variables
*The experience of a picker matters a lot at the beginning, then less and less: can you build a variable that represents that? Congestion?*

In [ ]:
# your code

#### Step 5 -- Tune
*Tune the 1-2 most promising models with `GridSearchCV`.*

In [ ]:
# your code

#### Step 6 -- Decide and test (once)
*Aggregate the predictions per shift and compute the staffing cost (helper below) of your model and of the current practice. Should you plan with a safety margin? Then evaluate **once** on the test period.*

In [ ]:
# your code

In [ ]:
# From task predictions to staffing (Step 6): number of pickers planned for each shift
PRODUCTIVE_MIN = 420          # productive minutes of one picker during a shift
COST_OVERTIME_H, COST_IDLE_H = 38.0, 24.0

def staffing_cost(tasks, predicted_minutes, margin=1.0):
    """tasks: DataFrame with columns date, shift, pick_minutes (true); predicted_minutes: one prediction per task.
    Plans ceil(margin x predicted workload / 420) pickers per shift and returns the cost (EUR) and a table per shift."""
    d = tasks[["date", "shift", "pick_minutes"]].assign(pred=np.asarray(predicted_minutes))
    s = d.groupby(["date", "shift"]).agg(true_min=("pick_minutes", "sum"), pred_min=("pred", "sum"))
    s["pickers"] = np.ceil(margin * s["pred_min"] / PRODUCTIVE_MIN)
    s["overtime_h"] = np.maximum(s["true_min"] - s["pickers"] * PRODUCTIVE_MIN, 0) / 60
    s["idle_h"] = np.maximum(s["pickers"] * PRODUCTIVE_MIN - s["true_min"], 0) / 60
    s["cost"] = COST_OVERTIME_H * s["overtime_h"] + COST_IDLE_H * s["idle_h"]
    return float(s["cost"].sum()), s

# the current practice: plan each shift with the average workload of that shift
avg_task = train_p.groupby("shift")["pick_minutes"].mean()
cost_now, _ = staffing_cost(train_p, train_p["shift"].map(avg_task))
print(f"current practice on the training period: {cost_now:,.0f} EUR")

In [ ]:
# your code

#### Step 7 -- Explain
*What drives the picking time? How many minutes does a newcomer lose? What did the new software change?*

In [ ]:
# your code

#### Step 8 -- Conclude
*Read the generator `make_picking`: did your models find the hidden rules? Limits?*

In [ ]:
# your code

---
# Situation 3 -- Rhodanis Parcel Hub: failures of conveyor motors
The hub sorts parcels with 90 conveyor motors equipped with sensors. An unplanned failure stops a line (about **6,000 EUR**:
repair, overtime, late parcels). Each week the maintenance team can **inspect 4 motors**; an inspection costs **300 EUR** and
prevents a failure in the next 14 days with probability **0.85**. Which motors should be inspected?

#### Step 1 -- Explore (EXAMPLE to extend)

In [ ]:
display(motors.head())
print(f"{motors['machine_id'].nunique()} motors, {motors['week'].nunique()} weeks; "
      f"share of motor-weeks followed by a failure within 14 days: {motors['failure_next_14d'].mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=motors, x="failure_next_14d", y="vibration_rms_mm_s", color=BLUE, fliersize=1, ax=axes[0])
axes[0].set_xticks([0, 1], ["no failure", "failure within 14 days"]); axes[0].set_xlabel("")
axes[0].set_title("Vibration level by outcome")
one = motors[motors["machine_id"] == "CV-01"]
axes[1].plot(one["week"], one["vibration_rms_mm_s"], color=BLUE)
axes[1].scatter(one.loc[one["failure_next_14d"] == 1, "week"], one.loc[one["failure_next_14d"] == 1, "vibration_rms_mm_s"],
                color=ORANGE, zorder=3, label="failure within 14 days")
axes[1].set_title("Motor CV-01: vibration over time"); axes[1].legend()
plt.tight_layout(); plt.show()

*How rare is a failure? How do the sensors behave before a failure? Are the motor models comparable (look at the vibration
level of each model when there is no failure)?*

In [ ]:
# your code

#### Step 2 -- Prepare (EXAMPLE to extend)
*One column reveals the future: which one? Why is a random split of the rows a bad idea here?*

In [ ]:
# Chronological split: the last 20 weeks are the test period, locked away until Step 6
last_weeks = np.sort(motors["week"].unique())[-20:]
train_m = motors[~motors["week"].isin(last_weeks)].copy()
test_m = motors[motors["week"].isin(last_weeks)].copy()
print(f"train: {len(train_m):,} motor-weeks  |  test: {len(test_m):,} motor-weeks")

# EXAMPLE of a first set of features
CAT3 = ["model", "zone"]
NUM3 = ["vibration_rms_mm_s", "motor_temp_max_c", "hours_since_maintenance"]

In [ ]:
# your code

#### Step 3 -- Benchmark (EXAMPLE to extend)
*Compare a naive baseline and the models of the course. Why is the accuracy useless here? Which metric would you use?*

In [ ]:
# EXAMPLE of a benchmark. The same motor appears many weeks: should two weeks of the same motor be in different folds?
# GroupKFold keeps all the weeks of a motor in the same fold.
cv = GroupKFold(n_splits=5)
models_3 = {
    "Logistic regression": make_pipeline(preprocess(CAT3, NUM3), LogisticRegression(max_iter=1000)),
    "Random forest": make_pipeline(preprocess(CAT3, NUM3), RandomForestClassifier(n_estimators=200, min_samples_leaf=10,
                                                                                 random_state=RANDOM_STATE)),
}
benchmark(models_3, train_m[CAT3 + NUM3], train_m["failure_next_14d"], cv,
          scoring=["roc_auc", "average_precision"], groups=train_m["machine_id"])

In [ ]:
# your code

#### Step 4 -- Improve the variables
*Sensors are more informative when compared with their own history: e.g. the change of vibration over the last 4 weeks (`groupby("machine_id")` + `shift`), or the vibration compared with the normal level of the motor model.*

In [ ]:
# your code

#### Step 5 -- Tune
*Tune the 1-2 most promising models (with `GroupKFold`).*

In [ ]:
# your code

#### Step 6 -- Decide and test (once)
*Build the weekly list of 4 motors from your best model and compare its net saving with the simple rule below. Then evaluate **once** on the test period.*

In [ ]:
# your code

In [ ]:
# Weekly inspection list (Step 6): the maintenance team can inspect at most 4 motors per week
CAPACITY, COST_INSPECTION, COST_FAILURE, INSPECTION_EFFECT = 4, 300.0, 6000.0, 0.85

def inspection_saving(df, score, capacity=CAPACITY):
    """Inspects, each week, the `capacity` motors with the highest `score`. Returns the net saving (EUR) and the list."""
    d = df.assign(score=np.asarray(score))
    chosen = d.sort_values("score", ascending=False).groupby("week").head(capacity)
    saving = INSPECTION_EFFECT * COST_FAILURE * chosen["failure_next_14d"].sum() - COST_INSPECTION * len(chosen)
    return float(saving), chosen

# a simple rule a maintenance manager could use without a model: inspect the 4 motors that vibrate the most
s, _ = inspection_saving(train_m, train_m["vibration_rms_mm_s"])
print(f"rule 'highest vibration' on the training period: net saving {s:,.0f} EUR")

In [ ]:
# your code

#### Step 7 -- Explain
*What does your model look at? Would the maintenance manager trust it?*

In [ ]:
# your code

#### Step 8 -- Conclude
*Read the generator `make_maintenance`: did your models find the hidden rules? Limits?*

In [ ]:
# your code